## **17. 데이터 엔지니어링 예제: Energy Star Score Prediction**

### **1. 데이터 로딩 및 포맷팅**

In [ ]:
import pandas as pd
import numpy as np

# 데이터프레임 로딩
df = pd.read_csv('data/ny_energy_2016.csv')
df.head()

In [ ]:
# 데이터프레임 요약 정보
df.info()

**[데이터 클리닝]** 데이터셋에서 결측값들이 "**Not Available**"의 텍스트로 기록됨 ${\rightarrow}$ pandas에서 인식 가능한 결측 값(```np.nan```)으로 변경

In [ ]:
# “Not Available” 값 포함 여부 출력
df[df == "Not Available"].any()

In [ ]:
# “Not Available” 값을 np.nan으로 대체
df.replace({'Not Available': np.nan}, inplace=True) 
df[df == "Not Available"].any()

**[데이터 포맷팅]** 수치형 변수들의 데이터 타입을 **float64**형으로 변경

In [ ]:
for col in list(df.columns):
    # 변수명에 수치 단위가 포함된 변수 선택
    if('ft²' in col or 
        'kBtu' in col or
        'Metric Tons CO2e' in col or
        'kWh' in col or
        'therms' in col or
        'gal' in col or
        'Score' in col):
            
            # 데이터 타입 변경
            df[col] = df[col].astype('float64')

df.info()

**[데이터 클리닝]** 1차 결측치 처리 ${\rightarrow}$ 결측 비율 높은 변수 제거

In [ ]:
# 변수 별 결측치 개수
ser_miss_cnt = df.isnull().sum().sort_values(ascending = False)
ser_miss_cnt

In [ ]:
# 변수 별 결측 비율
ser_miss_rate = ser_miss_cnt / len(df)
ser_miss_rate

In [ ]:
# 결측 비율이 50% 이상인 변수 제거
threshold = 0.5 # 변수 제거의 임계치
print(ser_miss_rate[ser_miss_rate > 0.5])

del_list = ser_miss_rate[ser_miss_rate > 0.5].index
print(f'제거 대상 변수: {del_list}')

In [ ]:
# 변수 제거
df.drop(columns=del_list, inplace=True)
df.head()

### **2. 탐색적 데이터 분석**

In [ ]:
# 시각화 옵션
%matplotlib
%config InlineBackend.figure_format = 'jpg'

from matplotlib import pyplot as plt
plt.rcParams['figure.figsize'] = [10, 5]
plt.ion() # 인터랙티브 시각화

import seaborn as sns
sns.set()  # 시각화 스타일

In [ ]:
# 변수명 간소화 (ENERGY STAR Score)
df = df.rename(columns = {'ENERGY STAR Score': 'Score'})

# Energy Star Score 히스토그램 시각화
plt.style.use('fivethirtyeight')
plt.hist(df['Score'].dropna(), bins = 100, edgecolor = 'k')
plt.xlabel('Score')
plt.ylabel('Number of Buildings')
plt.title('Energy Star Score Distribution')
plt.show()

In [ ]:
# Site EUI 히스토그램 시각화
plt.hist(df['Site EUI (kBtu/ft²)'].dropna(), bins = 20, edgecolor = 'black')
plt.xlabel('Site EUI')
plt.ylabel('Count')
plt.title('Site EUI Distribution')

Site EUI 히스토그램이 제대로 보이지 않음 😥 ${\rightarrow}$ 기술 통계량을 확인해보자

In [ ]:
# Site EUI 기술 통계
df['Site EUI (kBtu/ft²)'].describe()

In [ ]:
# 상위 10개의 Site EUI 값
df['Site EUI (kBtu/ft²)'].dropna().sort_values().tail(10)

In [ ]:
# 가장 큰 Site EUI 값의 빌딩
df.loc[8068]

**[데이터 클리닝]** Site EUI 값을 중심으로 한 IQR 기반 **이상치 제거**

In [ ]:
# Q1, Q3 계산
q1 = df['Site EUI (kBtu/ft²)'].quantile(0.25)
q3 = df['Site EUI (kBtu/ft²)'].quantile(0.75)

print('Site EUI, Q1 =', q1)
print('Site EUI, Q3 =', q3)

In [ ]:
# IQR (사분위수 범위)
iqr = q3-q1

"""
상한/하한 계산 (이상치 계수 3 적용)
 - 1.5: mild outliers
 - 3: extreme outliers
"""

upper_fence = q3 + 3*iqr # 상한
lower_fence = q1 - 3*iqr # 하한

print('IQR =', iqr)
print('Upper fence =', upper_fence)
print('Lower fence =', lower_fence)

In [ ]:
# 이상치 제거
df = df[ (df['Site EUI (kBtu/ft²)'] < upper_fence) &
      (df['Site EUI (kBtu/ft²)'] > lower_fence) ]

df['Site EUI (kBtu/ft²)'].shape

In [ ]:
# Site EUI 히스토그램 시각화 (이상치 제거 후)

plt.hist(df['Site EUI (kBtu/ft²)'].dropna(), bins = 20, edgecolor = 'black')
plt.xlabel('Site EUI')
plt.ylabel('Count')
plt.title('Site EUI Distribution')

**[데이터 탐색]** 대상 변수(Energy Star Score)와 범주형 변수와의 상관관계

범주형 변수1: **Largest Property Use Type** (주 용도에 따른 빌딩 유형)

In [ ]:
# 빌딩 유형과의 상관관계
df_has_score = df.dropna(subset=['Score'])  # 스코어가 있는 데이터만 선택
types = df_has_score['Largest Property Use Type'].value_counts() # 빌딩 유형별 빈도 계산
types

In [ ]:
# 빈도 100 이상의 빌딩 유형
types_100 = types[types.values > 100].index
types_100

In [ ]:
# 빌딩 유형 히스토그램 시각화

for b_type in types_100:
    subset = df[df['Largest Property Use Type'] == b_type]
    sns.kdeplot(subset['Score'].dropna(),
               label=b_type, alpha=0.8)

plt.xlabel('Energy Star Scores')
plt.ylabel('Density')
plt.title('Density Plot of Energy Star Scores by Building Type')
plt.legend(loc='best')

범주형 변수2: **Borough** (빌딩 자치구 위치)

In [ ]:
boroughs = df_has_score['Borough'].value_counts()  # 자치구별 빈도 계샨
print(boroughs)

boroughs = boroughs.index
print(boroughs)

In [ ]:
# 자치구 히스토그램 시각화
for borough in boroughs:
    subset = df[df['Borough'] == borough]
    sns.kdeplot(subset['Score'].dropna(),
               label=borough)

plt.xlabel("Energy Star Scores")
plt.ylabel('Density')
plt.title("Density Plot of Energy Star Scores by Borough")
plt.legend(loc='best')
    

**[데이터 탐색]** 대상 변수(Energy Star Score)와 수치형 변수와의 상관관계

In [ ]:
# 수치형 데이터만 선택
numeric_df = df.select_dtypes(include=['int64', 'float64'])
numeric_df.head()

In [ ]:
# 대상변수와의 상관계수
corr = numeric_df.corr()['Score'].sort_values()
corr

In [ ]:
"""
Two-Variable Plots
- Variable 1: Site EUI
- Variable 2: Largest Property Use Type
- Target: Energy Star Score
"""

# Extract the building types
df['Largest Property Use Type'] = df.dropna(subset = ['Score'])['Largest Property Use Type']

In [ ]:
# Limit to building types with more than 100 observations
print(types_100)
df = df[df['Largest Property Use Type'].isin(types_100)]
df.shape

In [ ]:
# Scatterplot of Score vs Site EUI

sns.lmplot(
    data=df,
    x='Site EUI (kBtu/ft²)', 
    y='Score',
    hue='Largest Property Use Type',
    scatter_kws={'alpha': 0.8}, 
    fit_reg=False,    
    aspect=1.2
)

plt.xlabel('Site EUI')
plt.ylabel('Enery Star Score')
plt.title('Energy Star Score vs Site EUI')

In [ ]:
"""
Pairs Plot
"""

# Select the columns to plot
df_plot = df[['Score',
              'Site EUI (kBtu/ft²)',
              'Weather Normalized Site EUI (kBtu/ft²)',
              'Weather Normalized Source EUI (kBtu/ft²)',
              'Source EUI (kBtu/ft²)',
              'Weather Normalized Site Electricity Intensity (kWh/ft²)',
              'Total GHG Emissions (Metric Tons CO2e)']]

In [ ]:
# Rename columns
df_plot.rename(columns={'Site EUI (kBtu/ft²)':
                        'Site EUI',
                       'Weather Normalized Site EUI (kBtu/ft²)':
                        'Weather Norm Site EUI',
                       'Weather Normalized Source EUI (kBtu/ft²)':
                        'Weather Norm Src EUI)',
                       'Source EUI (kBtu/ft²)':
                        'Source EUI',
                       'Weather Normalized Site Electricity Intensity (kWh/ft²)':
                        'Weather Norm Site Elec Intensity',
                       'Total GHG Emissions (Metric Tons CO2e)':
                       'Total GHG Emissions'}, inplace=True)

In [ ]:
# Replace the inf with nan
df_plot = df_plot.replace({np.inf: np.nan, -np.inf: np.nan})

# Drop NA values
df_plot.dropna(inplace=True)

In [ ]:
# Function to calculate correlation coefficient between two columns

# Plot a correlation matrix using pairplot
sns.pairplot(df_plot, kind='reg', diag_kind='kde', 
            plot_kws={'scatter_kws': {'alpha': 0.1}})


In [ ]:
"""
Feature Extraction & Selection
"""

# Create columns with square root and log of numeric columns

df_numeric = df.select_dtypes('number') # select the numeric columns

for col in df_numeric.columns:
    # Skip the Energy Star Score column
    if col == 'Score':
        continue
    else:
        df_numeric['sqrt_' + col] = np.sqrt(df_numeric[col])
        df_numeric['log_' + col] = np.log(df_numeric[col])


In [ ]:
df_numeric.head()

In [ ]:
# One-hot encoding for the categorical columns

df_categoric = df[['Borough', 'Largest Property Use Type']]  # Select the categorical columns
df_categoric.head()

In [ ]:
df_categoric = pd.get_dummies(df_categoric)  # One-hot encoding
df_categoric.head()

In [ ]:
# Join the two DataFrames using concat
df_transformed = pd.concat([df_numeric, df_categoric], axis=1)

# Drop buildings without an Energy Star Score
df_transformed.dropna(subset=['Score'], inplace=True)

In [ ]:
df_transformed.shape

In [ ]:
df_transformed.info()

In [ ]:
"""
Remove Collinear Features
"""

df_plot = df[['Weather Normalized Site EUI (kBtu/ft²)', 
              'Site EUI (kBtu/ft²)']].dropna()

plt.plot(df_plot['Site EUI (kBtu/ft²)'], 
         df_plot['Weather Normalized Site EUI (kBtu/ft²)'], 
         'bo')
plt.xlabel('Site EUI')
plt.ylabel('Weather Norm EUI')
plt.title('Weather Norm EUI vs Site EUI, R = %0.4f' % 
          np.corrcoef(df[['Weather Normalized Site EUI (kBtu/ft²)',
                          'Site EUI (kBtu/ft²)']].dropna(), rowvar=False)[0][1]);

In [ ]:
def remove_collinear_features(x, threshold):
    '''
    Objective:
        Remove collinear features in a dataframe with a correlation coefficient
        greater than the threshold. Removing collinear features can help a model
        to generalize and improves the interpretability of the model.
        
    Inputs: 
        threshold: any features with correlations greater than this value are removed
    
    Output: 
        dataframe that contains only the non-highly-collinear features
    '''
    
    # Dont want to remove correlations between Energy Star Score
    y = x['Score']
    x = x.drop(columns = ['Score'])
    
    # Calculate the correlation matrix
    corr_matrix = x.corr()
    iters = range(len(corr_matrix.columns) - 1)
    drop_cols = []

    # Iterate through the correlation matrix and compare correlations
    for i in iters:
        for j in range(i):
            item = corr_matrix.iloc[j:(j+1), (i+1):(i+2)]
            col = item.columns
            row = item.index
            val = abs(item.values)
            
            # If correlation exceeds the threshold
            if val >= threshold:
                # Print the correlated features and the correlation value
                # print(col.values[0], "|", row.values[0], "|", round(val[0][0], 2))
                drop_cols.append(col.values[0])

    # Drop one of each pair of correlated columns
    drops = set(drop_cols)
    x = x.drop(columns = drops)
    x = x.drop(columns = ['Weather Normalized Site EUI (kBtu/ft²)', 
                          'Water Use (All Water Sources) (kgal)',
                          'log_Water Use (All Water Sources) (kgal)',
                          'Largest Property Use Type - Gross Floor Area (ft²)'])
    
    # Add the score back in to the data
    x['Score'] = y
               
    return x

In [ ]:
# Remove the collinear features above a specified correlation coefficient
collinear_threshold = 0.6
features = remove_collinear_features(df_transformed, collinear_threshold)

In [ ]:
# Remove any columns with all na values
features.dropna(axis=1, how = 'all', inplace=True)
features.shape

In [ ]:

features.info()

In [ ]:
"""
Split Into Training and Testing Sets
"""

# Splitting data into training and testing
from sklearn.model_selection import train_test_split

# Separate out the features and targets
targets = pd.DataFrame(features['Score']) # y: vector
features = features.drop(columns='Score') # X: matrix

# Replace the inf and -inf with nan (required for later imputation)
features = features.replace({np.inf: np.nan, -np.inf: np.nan})

# Handle the missing values with imputations
features.interpolate(inplace=True)
features.bfill(inplace=True)

# Split into 70% training and 30% testing set
X_train, X_test, y_train, y_test = train_test_split(
    features, targets, test_size = 0.3, random_state = 42)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape) # 학습 시 정답지
print(y_test.shape) # 테스트 시 정답

In [ ]:
# Train the model

from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor()
model.fit(X_train, y_train)

In [ ]:
# Predict scores
y_hat = model.predict(X_test) 
print(y_test.values.reshape(-1), y_hat)

In [ ]:
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(y_test, y_hat)

In [ ]:
print("Baseline Performance on the test set: MAE = %0.4f" % mae)

In [ ]:
plt.plot(y_test.values.reshape(-1), y_hat, 'bo')
plt.xlabel("Actual")
plt.ylabel("Prediction")